In [1]:
using JLD2
using Printf
using StaticArrays
using Statistics
using WaveGreen2D.Chebyshev: ChebyshevSeries, TransformedChebyshevSeries, gradient, hessian, contains

In [2]:
@load "../src/chebyshev_series.jld2" L₁_series L₂_series;

In [3]:
function u(x::SVector{3,Float64})
    A, B, H = x

    return SA[A, B, log(H)]
end


function ∇u(x::SVector{3,Float64})
    _, _, H = x

    grad_u = one(MMatrix{3, 3, Float64})
    grad_u[3, 3] = 1/H
    
    return SMatrix(grad_u)
end


function Hu(x::SVector{3,Float64})
    _, _, H = x

    hess_u = zero(MArray{Tuple{3, 3, 3}, Float64, 3})
    hess_u[3, 3, 3] = -1/H^2

    return SArray(hess_u)
end;

In [4]:
C₁1 = L₁_series[1]
C₁2 = TransformedChebyshevSeries(L₁_series[2], u, ∇u, Hu)
C₁3 = TransformedChebyshevSeries(L₁_series[3], u, ∇u, Hu)

C₂1 = TransformedChebyshevSeries(L₂_series[1], u, ∇u, Hu)
C₂2 = TransformedChebyshevSeries(L₂_series[2], u, ∇u, Hu)
C₂3 = TransformedChebyshevSeries(L₂_series[3], u, ∇u, Hu)
C₂4 = TransformedChebyshevSeries(L₂_series[4], u, ∇u, Hu)

function C₁(x::SVector{3, Float64})
    if contains(C₁1, x)
        return C₁1(x)
    elseif contains(C₁2, x)
        return C₁2(x)
    elseif contains(C₁3, x)
        return C₁3(x)
    else
        throw(DomainError(x))
    end
end


function C₁_gradient(x::SVector{3, Float64})
    if contains(C₁1, x)
        return gradient(C₁1, x)
    elseif contains(C₁2, x)
        return gradient(C₁2, x)
    elseif contains(C₁3, x)
        return gradient(C₁3, x)
    else
        throw(DomainError(x))
    end
end


function C₁_hessian(x::SVector{3, Float64})
    if contains(C₁1, x)
        return hessian(C₁1, x)
    elseif contains(C₁2, x)
        return hessian(C₁2, x)
    elseif contains(C₁3, x)
        return hessian(C₁3, x)
    else
        throw(DomainError(x))
    end
end


function C₂(x::SVector{3, Float64})
    if contains(C₂1, x)
        return C₂1(x)
    elseif contains(C₂2, x)
        return C₂2(x)
    elseif contains(C₂3, x)
        return C₂3(x)
    elseif contains(C₂4, x)
        return C₂4(x)
    else
        throw(DomainError(x))
    end
end


function C₂_gradient(x::SVector{3, Float64})
    if contains(C₂1, x)
        return gradient(C₂1, x)
    elseif contains(C₂2, x)
        return gradient(C₂2, x)
    elseif contains(C₂3, x)
        return gradient(C₂3, x)
    elseif contains(C₂4, x)
        return gradient(C₂4, x)
    else
        throw(DomainError(x))
    end
end


function C₂_hessian(x::SVector{3, Float64})
    if contains(C₂1, x)
        return hessian(C₂1, x)
    elseif contains(C₂2, x)
        return hessian(C₂2, x)
    elseif contains(C₂3, x)
        return hessian(C₂3, x)
    elseif contains(C₂4, x)
        return hessian(C₂4, x)
    else
        throw(DomainError(x))
    end
end;

In [5]:
@load "../test/data/l1_test_data.jld2" x1 L1 ∇L1 HL1;

In [25]:
npoints = length(x1)

C1 = Vector{Float64}(undef, npoints)
C1g = Vector{Float64}(undef, npoints)
C1h = Vector{Float64}(undef, npoints)
∇C1 = Matrix{Float64}(undef, npoints, 3)
∇C1h = Matrix{Float64}(undef, npoints, 3)
HC1 = Array{Float64, 3}(undef, npoints, 3, 3);

In [26]:
for i in 1:npoints
    y₁ = C₁(x1[i])
    y₂, ∇y₂ = C₁_gradient(x1[i])
    y₃, ∇y₃, Hy₃ = C₁_hessian(x1[i])

    C1[i] = y₁
    C1g[i] = y₂
    C1h[i] = y₃
    ∇C1[i, :] .= ∇y₂
    ∇C1h[i, :] .= ∇y₃
    HC1[i, :, :] .= Hy₃
end

In [11]:
err_L = @. abs(L1 - C1)
err_∇L = @. abs(∇L1 - ∇C1)
err_HL = @. abs(HL1 - HC1)

mean_err_L = mean(err_L)
mean_err_∇L = dropdims(mean(err_∇L; dims=1); dims=1)
mean_err_HL = dropdims(mean(err_HL; dims=1); dims=1)

std_err_L = std(err_L)
std_err_∇L = dropdims(std(err_∇L; dims=1); dims=1)
std_err_HL = dropdims(std(err_HL; dims=1); dims=1)

lim_err_L = mean_err_L + 3 * std_err_L
lim_err_∇L = reshape(mean_err_∇L + 3 * std_err_∇L, 1, 3)
lim_err_HL = reshape(mean_err_HL + 3 * std_err_HL, 1, 3, 3)

pct_err_L = sum(err_L .< lim_err_L) * 100 / npoints
pct_err_∇L = dropdims(count(err_∇L .< lim_err_∇L; dims=1); dims=1) * 100 / npoints
pct_err_HL = dropdims(sum(err_HL .< lim_err_HL; dims=1); dims=1) * 100 / npoints;

In [12]:
result = """
L₁c mean absolute error = $(@sprintf("%.0e", mean_err_L))

∇L₁c mean absolute error = $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_∇L...))

HL₁c mean absolute error = $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_HL[1, :]...))
                           $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_HL[2, :]...))
                           $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_HL[3, :]...))

For L₁c, ∇L₁c and HL₁c, the error value for each point is compared with a corresponding reference
value. The reference value is the mean absolute error plus three times the standard deviation. The
following results display the percentage of points for which the error is less than the reference.

Percentage of L₁c error values lower than reference = $(@sprintf("%.1f%%", pct_err_L))

Percentage of ∇L₁c error values lower than reference = $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_∇L...))

Percentage of HL₁c error values lower than reference = $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_HL[1, :]...))
                                                       $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_HL[2, :]...))
                                                       $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_HL[3, :]...))
"""

println(result)

L₁c mean absolute error = 6e-16

∇L₁c mean absolute error = [2e-14  1e-14  7e-15]

HL₁c mean absolute error = [2e-12  6e-13  3e-13]
                           [6e-13  9e-13  2e-13]
                           [3e-13  2e-13  3e-13]

For L₁c, ∇L₁c and HL₁c, the error value for each point is compared with a corresponding reference
value. The reference value is the mean absolute error plus three times the standard deviation. The
following results display the percentage of points for which the error is less than the reference.

Percentage of L₁c error values lower than reference = 99.2%

Percentage of ∇L₁c error values lower than reference = [97.8%  98.2%  99.2%]

Percentage of HL₁c error values lower than reference = [97.9%  98.5%  98.5%]
                                                       [98.5%  98.7%  98.5%]
                                                       [98.5%  98.5%  99.2%]



In [18]:
maximum(abs.(L1 - C1))

3.3306690738754696e-15

In [17]:
maximum(abs.(∇L1 - ∇C1))

3.480549182199866e-13

In [20]:
maximum(abs.(HL1 - HC1))

5.711520145723625e-11

In [21]:
maximum(abs.(C1 - C1g))

0.0

In [22]:
maximum(abs.(C1 - C1h))

0.0

In [27]:
maximum(abs.(∇C1 - ∇C1h))

0.0

In [28]:
∇C1 == ∇C1h

true